# OCDD: train the second model (ConvNeXt-Tiny) on Kaggle
Before running: right side **Session options > Accelerator = GPU (P100 or T4)**, **Persistence = Files only**. Add your uploaded dataset (**Add Input**). Then **Run All**. At the end download `run8_out` from the Output tab.

In [ ]:
import os, glob, shutil, subprocess, sys
inp = "/kaggle/input"
manifest = None; cache = None
for d, dirs, files in os.walk(inp):
    if manifest is None and "manifest_v4.csv" in files: manifest = os.path.join(d, "manifest_v4.csv")
    if cache is None and sum(f.endswith(".jpg") for f in files[:20]) >= 10: cache = d
    if manifest and cache: break
assert manifest and cache, "dataset not found: use Add Input and pick your OCDD dataset"
code_dir = os.path.dirname(manifest)
print("manifest:", manifest); print("cache   :", cache, "(%d files)" % len(os.listdir(cache)))
ROOT = "/kaggle/working/ocdd"
os.makedirs(ROOT + "/data/interim_v4", exist_ok=True); os.makedirs(ROOT + "/data/interim", exist_ok=True)
shutil.copy(manifest, ROOT + "/data/interim_v4/manifest.csv")
if not os.path.exists(ROOT + "/data/interim/cache256"): os.symlink(cache, ROOT + "/data/interim/cache256")
for f in ("train.py", "ctl.py", "progress.py"): shutil.copy(os.path.join(code_dir, f), ROOT)
os.environ["OCDD_ROOT"] = ROOT
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - turn the accelerator on")

In [ ]:
# ConvNeXt-Tiny, same data and split as run7 (EfficientNet-V2-S). Roughly 1-2 hours on a P100.
cmd = [sys.executable, "-u", "train.py", "--root", ROOT, "--arch", "convnext_tiny", "--interim", "interim_v4", "--epochs", "15",
       "--batch", "64", "--lr", "2e-4", "--sampling-power", "0.75", "--out", "run8"]
p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in p.stdout:
    print(line, end="")
p.wait(); print("exit code", p.returncode)


In [ ]:
out = "/kaggle/working/run8_out"; os.makedirs(out, exist_ok=True)
for f in ("best.pt", "classes.json", "temperature.json", "report.txt"):
    s = os.path.join(ROOT, "training", "run8", f)
    if os.path.exists(s): shutil.copy(s, out)
print(os.listdir(out)); print(open(out + "/report.txt").read()[:1500])